In [5]:
import requests
from rapidfuzz import fuzz

LOBID = "https://lobid.org/gnd/search"

def normalize(s: str) -> str:
    return (
        s.lower()
        .replace("–", "-")
        .replace("—", "-")
        .replace(",", " ")
        .strip()
    )

def gnd_lookup(literal: str, entity_type: str | None = None, limit: int = 10):
    params = {
        "q": literal,
        "format": "json",
        "size": limit,
    }
    if entity_type:
        params["type"] = entity_type

    r = requests.get(LOBID, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    query_norm = normalize(literal)

    print(query_norm)
    candidates = []

    for item in data.get("member", []):
        preferred = item.get("preferredName") or item.get("preferredNameForThePerson") or ""
        variants = item.get("variantName", [])
        if isinstance(variants, str):
            variants = [variants]

        labels = [preferred] + variants
        label_scores = [
            fuzz.token_sort_ratio(query_norm, normalize(label)) / 100
            for label in labels
            if label
        ]

        lexical_score = max(label_scores) if label_scores else 0

        candidates.append({
            "gnd_id": item.get("gndIdentifier"),
            "uri": item.get("id"),
            "preferred_name": preferred,
            "type": item.get("type"),
            "score": round(lexical_score, 3),
            "raw": item,
        })

    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates

def decide_gnd_match(literal: str, entity_type: str | None = None):
    candidates = gnd_lookup(literal, entity_type=entity_type)

    if not candidates:
        return {
            "literal": literal,
            "decision": "no_candidate",
            "gnd_id": None,
            "confidence": 0.0,
            "candidates": [],
        }

    top = candidates[0]
    second = candidates[1] if len(candidates) > 1 else None

    margin = top["score"] - second["score"] if second else top["score"]

    if top["score"] >= 0.90 and margin >= 0.08:
        decision = "match"
    elif top["score"] >= 0.70:
        decision = "ambiguous_candidate"
    else:
        decision = "no_safe_match"

    return {
        "literal": literal,
        "decision": decision,
        "gnd_id": top["gnd_id"] if decision != "no_safe_match" else None,
        "preferred_name": top["preferred_name"] if decision != "no_safe_match" else None,
        "confidence": top["score"],
        "candidates": candidates[:5],
    }

print(decide_gnd_match("Bayern Tourismus Marketing GmbH"))

bayern tourismus marketing gmbh
{'literal': 'Bayern Tourismus Marketing GmbH', 'decision': 'ambiguous_candidate', 'gnd_id': '1066719632', 'preferred_name': 'Franken Tourismus Marketing GmbH', 'confidence': 0.889, 'candidates': [{'gnd_id': '1066719632', 'uri': 'https://d-nb.info/gnd/1066719632', 'preferred_name': 'Franken Tourismus Marketing GmbH', 'type': ['CorporateBody', 'AuthorityResource', 'Company'], 'score': 0.889, 'raw': {'type': ['CorporateBody', 'AuthorityResource', 'Company'], '@context': 'https://lobid.org/gnd/context.jsonld', 'gndSubjectCategory': [{'id': 'https://d-nb.info/standards/vocab/gnd/gnd-sc#10.6b', 'label': 'Fremdenverkehr, Hotel- und Gaststättengewerbe'}, {'id': 'https://d-nb.info/standards/vocab/gnd/gnd-sc#10.12b', 'label': 'Einzelne Branchen der Industrie und des Handwerks'}], 'geographicAreaCode': [{'id': 'https://d-nb.info/standards/vocab/gnd/geographic-area-code#XA-DE-BY', 'label': 'Bayern'}], 'placeOfBusiness': [{'id': 'https://d-nb.info/gnd/4042742-0', 'la